# 01 — Data Collection

**Purpose:** Download and parse the raw Shona text sources that feed the TauraBot corpus.

**Expected runtime (current sections):** ~1 minute (Bible + small HF datasets).

**Outputs (`data/raw/` and `data/eval/`):**
- `bible/bible_verses.txt` — 31K Bible verses (one per line)
- `wikipedia/wiki_text.txt` — 115K paragraphs from 11.6K Shona Wikipedia articles
- `masakhane/ner_text.txt` — 8.9K Shona sentences from MasakhaNER 2.0
- `masakhane/news_text.txt` — 1.8K Shona news headlines + bodies
- `eval/sib200_sna.txt` — 1K held-out eval sentences from SIB-200

**Sources status:**

| # | Source | License | Tier | Status |
|---|---|---|---|---|
| 1 | Shona Bible (BDRSC) | CC BY-SA 4.0 | small | ✅ |
| 2 | Wikipedia Shona | CC BY-SA 3.0 | small | ✅ |
| 3 | Masakhane NER 2.0 | AFL-3.0 | small | ✅ |
| 4 | Masakhane NEWS | AFL-3.0 | small | ✅ |
| 5 | SIB-200 (held-out eval) | CC BY-SA 4.0 | small | ✅ |
| 6 | mC4 (`allenai/c4` sn) | ODC-BY | large | ⏳ next step |
| 7 | GlotCC-V1 | CC0 | large | ⏳ next step |
| 8 | HPLT 2.0 cleaned | CC0 | large | ⏳ next step |
| 9 | OPUS / JW300 | CC BY-SA | medium | ⏳ later |
| – | FLORES-Plus eval | CC BY-SA 4.0 | small | ⏸ gated on HF |
| – | BBC Shona | © BBC | – | ❌ dropped (license) |

**License posture:** all sources here are compatible with CC BY-SA 4.0 — the license the released corpus + model will carry.

## Setup

Configure logging and load the project config. Run from the project root.

In [1]:
import logging
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True,
)
logger = logging.getLogger("01_data_collection")
logger.info("project root: %s", PROJECT_ROOT)

2026-05-24 13:58:53,020 | INFO | 01_data_collection | project root: /Users/apple/Desktop/taurabot


In [2]:
from src.data.bible_downloader import BibleDownloader, load_config
from src.data.hf_source import HFDatasetSource, load_specs

cfg = load_config(PROJECT_ROOT / "configs/config.yaml")

2026-05-24 13:58:54,624 | INFO | datasets | PyTorch version 2.2.2 available.


## Source 1 — Shona Bible (BDRSC)

**Biblica® Bhaibheri Dzvene Rakasununguka MuChiShona Chanhasi 2017** — full OT + NT, 1,189 chapters, pre-cleaned plain-text. Downloaded as a single zip from eBible.org.

**License: CC BY-SA 4.0** — this is what locks in the corpus + model license for the whole project. The `Biblica®` trademark stays out of derivative outputs; we only extract verse text.

In [3]:
bible_cfg = cfg["sources"]["bible"]
print(f"Source:      {bible_cfg['name']}")
print(f"License:     {bible_cfg['license']}")
print(f"Attribution: {bible_cfg['attribution']}")

dl = BibleDownloader(bible_cfg, cfg["paths"]["raw_dir"])
dl.download()        # idempotent: skips if already on disk
dl.extract()         # idempotent: skips if already extracted
records = list(dl.parse_all())

verses_path = dl.write_verses_corpus(records, PROJECT_ROOT / bible_cfg["verses_file"])
jsonl_path  = dl.write_records_jsonl(records, PROJECT_ROOT / "data/raw/bible/bible_chapters.jsonl")

2026-05-24 13:58:55,096 | INFO | src.data.bible_downloader | Zip already present at /Users/apple/Desktop/taurabot/notebooks/data/raw/bible/sna_readaloud.zip — skipping download.


2026-05-24 13:58:55,099 | INFO | src.data.bible_downloader | Already extracted at /Users/apple/Desktop/taurabot/notebooks/data/raw/bible/sna_readaloud — skipping.


2026-05-24 13:58:55,110 | INFO | src.data.bible_downloader | Parsing 1190 chapter files


Source:      Biblica Bhaibheri Dzvene Rakasununguka MuChiShona Chanhasi 2017 (BDRSC)
License:     CC BY-SA 4.0
Attribution: © 2005, 2018 Biblica, Inc. — https://www.biblica.com


2026-05-24 13:58:55,323 | INFO | src.data.bible_downloader | Wrote 31103 verses to /Users/apple/Desktop/taurabot/data/raw/bible/bible_verses.txt


2026-05-24 13:58:55,415 | INFO | src.data.bible_downloader | Wrote 1189 chapter records to /Users/apple/Desktop/taurabot/data/raw/bible/bible_chapters.jsonl


In [4]:
# Bible sanity check: per-testament counts
OT_END = "039"
ot_books = {(r.book_num, r.book_abbr) for r in records if r.book_num <= OT_END}
nt_books = {(r.book_num, r.book_abbr) for r in records if r.book_num > OT_END}
print(f"OT books: {len(ot_books)}  NT books: {len(nt_books)}  Total: {len(ot_books)+len(nt_books)}")
print(f"First verse: {records[0].verses[0]}")
print(f"Last verse:  {records[-1].verses[-1]}")

OT books: 38  NT books: 28  Total: 66
First verse: Pakutanga Mwari akasika matenga nenyika.
Last verse:  Nyasha dzaIshe Jesu ngadzive navanhu vaMwari. Ameni.


## Sources 2–5 — Small HuggingFace bundle

All small-tier HF datasets are loaded by the same generic `HFDatasetSource` class — defined in [`src/data/hf_source.py`](../src/data/hf_source.py). Each spec is read from `configs/config.yaml`.

| Source | Repo | Config | Why |
|---|---|---|---|
| Wikipedia Shona | `wikimedia/wikipedia` | `20231101.sn` | Encyclopedic register; native Shona authors |
| MasakhaNER 2.0 | `masakhane/masakhaner2` | `sna` | Native-speaker-curated news sentences |
| MasakhaNEWS | `masakhane/masakhanews` | `sna` | Long-form Shona news articles |
| **SIB-200 (eval)** | `Davlan/sib200` | `sna_Latn` | Held-out test set — NEVER in training |

In [5]:
small_specs = load_specs(cfg, only_tiers={"small"})
print(f"Loading {len(small_specs)} small HF sources:\n")
for s in small_specs:
    flag = "  [HELD-OUT EVAL]" if s.held_out else ""
    print(f"  - {s.name:18} repo={s.repo:35} config={s.config or '-':12}{flag}")

Loading 4 small HF sources:

  - wikipedia          repo=wikimedia/wikipedia                 config=20231101.sn 
  - masakhane_ner      repo=masakhane/masakhaner2               config=sna         
  - masakhane_news     repo=masakhane/masakhanews               config=sna         
  - sib200_eval        repo=Davlan/sib200                       config=sna_Latn      [HELD-OUT EVAL]


In [6]:
for spec in small_specs:
    HFDatasetSource(spec, PROJECT_ROOT).download()

2026-05-24 13:58:55,472 | INFO | src.data.hf_source | [wikipedia] output already exists at /Users/apple/Desktop/taurabot/data/raw/wikipedia/wiki_text.txt (115575 lines) — skipping. Pass force=True to refresh.


2026-05-24 13:58:55,475 | INFO | src.data.hf_source | [masakhane_ner] output already exists at /Users/apple/Desktop/taurabot/data/raw/masakhane/ner_text.txt (8867 lines) — skipping. Pass force=True to refresh.


2026-05-24 13:58:55,491 | INFO | src.data.hf_source | [masakhane_news] output already exists at /Users/apple/Desktop/taurabot/data/raw/masakhane/news_text.txt (3684 lines) — skipping. Pass force=True to refresh.


2026-05-24 13:58:55,494 | INFO | src.data.hf_source | [sib200_eval] output already exists at /Users/apple/Desktop/taurabot/data/eval/sib200_sna.txt (1004 lines) — skipping. Pass force=True to refresh.


## Phase 1 corpus summary so far

In [7]:
# Tally lines + words across every raw corpus file we've produced.
# Excludes data/eval/ — those are held-out, not part of the training corpus.
raw_root = PROJECT_ROOT / "data/raw"
training_files = sorted(p for p in raw_root.rglob("*.txt"))

rows = []
tot_lines = tot_words = tot_chars = 0
for path in training_files:
    with path.open(encoding="utf-8") as fh:
        n_lines = n_words = n_chars = 0
        for ln in fh:
            n_lines += 1
            n_words += len(ln.split())
            n_chars += len(ln)
    rows.append((path.relative_to(PROJECT_ROOT), n_lines, n_words, n_chars))
    tot_lines += n_lines
    tot_words += n_words
    tot_chars += n_chars

print(f"{'source file':50} {'lines':>10} {'words':>12} {'MB':>8}")
print("-" * 84)
for path, lines, words, chars in rows:
    print(f"{str(path):50} {lines:>10,} {words:>12,} {chars/1024/1024:>8.1f}")
print("-" * 84)
print(f"{'TOTAL (training corpus, before cleaning)':50} {tot_lines:>10,} {tot_words:>12,} {tot_chars/1024/1024:>8.1f}")

source file                                             lines        words       MB
------------------------------------------------------------------------------------
data/raw/bible/bible_verses.txt                        31,103      477,485      3.7
data/raw/bible/sna_readaloud/sna_000_000_000_read.txt          4           33      0.0
data/raw/bible/sna_readaloud/sna_002_GEN_01_read.txt         33          514      0.0
data/raw/bible/sna_readaloud/sna_002_GEN_02_read.txt         27          375      0.0
data/raw/bible/sna_readaloud/sna_002_GEN_03_read.txt         26          409      0.0
data/raw/bible/sna_readaloud/sna_002_GEN_04_read.txt         28          391      0.0
data/raw/bible/sna_readaloud/sna_002_GEN_05_read.txt         34          404      0.0
data/raw/bible/sna_readaloud/sna_002_GEN_06_read.txt         24          353      0.0
data/raw/bible/sna_readaloud/sna_002_GEN_07_read.txt         26          349      0.0
data/raw/bible/sna_readaloud/sna_002_GEN_08_read.txt      

## What's next

**Step gated by user confirmation** — the three large web crawls (mC4, GlotCC-V1, HPLT 2.0) collectively download ~1–2 GB of Shona text. They go in a separate notebook section so you can decide when to commit the disk + bandwidth.

After all sources are gathered, the cleaning pipeline (`src/data/cleaner.py`) will:
1. Deduplicate across sources (essential — mC4/GlotCC/HPLT all derive from Common Crawl and will overlap heavily)
2. Run language ID to drop non-Shona contamination
3. Apply length + character-ratio filters
4. Write the unified `data/processed/corpus.txt`

Then `notebooks/03_corpus_stats.ipynb` produces the stats that go in the README / dataset card.